# DocFusion: Extraction Experiments

This notebook documents the field extraction pipeline, showing how OCR output
is processed to extract vendor, date, and total fields using regex + spatial heuristics.
It includes examples of success cases, failure cases, and the preprocessing improvements.

In [ ]:
import sys
import os
import re
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = str(Path(".").resolve().parent)
sys.path.insert(0, PROJECT_ROOT)

from ocr_engine import OCREngine, OCRResult
from field_extractor import FieldExtractor

## 1. OCR Preprocessing Pipeline

Before OCR, each image passes through:
1. **CLAHE** (Contrast Limited Adaptive Histogram Equalization) — enhances local contrast
2. **fastNlMeansDenoising** — removes noise while preserving edges
3. **Downscaling** to 1536px max side — balances resolution vs speed

This dramatically improves OCR quality on low-contrast or noisy scans.

In [ ]:
# Demonstrate preprocessing on a sample image
def show_preprocessing(image_bgr):
    """Show original vs preprocessed side by side."""
    preprocessed = OCREngine._preprocess(image_bgr)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original", fontsize=12)
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(preprocessed, cv2.COLOR_BGR2RGB))
    axes[1].set_title("After CLAHE + Denoise", fontsize=12)
    axes[1].axis("off")
    plt.suptitle("Preprocessing Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Create a synthetic noisy receipt for demonstration
np.random.seed(42)
demo = np.ones((400, 300, 3), dtype=np.uint8) * 200
noise = np.random.normal(0, 30, demo.shape).astype(np.int16)
demo = np.clip(demo.astype(np.int16) + noise, 0, 255).astype(np.uint8)
cv2.putText(demo, "ACME STORE", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (20, 20, 20), 2)
cv2.putText(demo, "Date: 2024-03-15", (30, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (40, 40, 40), 1)
cv2.putText(demo, "Total: $42.50", (30, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (30, 30, 30), 1)

show_preprocessing(demo)

## 2. Extraction Rules

The `FieldExtractor` uses three strategies for each field:

### Vendor
- Takes the first non-numeric, non-keyword line from the top of the receipt
- Merges short consecutive header lines (e.g., "Berg" + "hotel" → "Berghotel")
- Skips lines matching keywords: tax, receipt, invoice, tel, fax, etc.
- Filters lines with confidence < 0.25

### Date
- Tries 4 regex patterns: DD/MM/YYYY, YYYY-MM-DD, DD Mon YYYY, Mon DD YYYY
- Falls back to `dateutil.parser` for fuzzy parsing

### Total
1. **Keyword search**: Finds "total", "grand total", etc. and extracts money from that line or the next 2 lines
2. **Bottom-half scan**: Finds the largest amount in the bottom 50% of lines
3. **Global fallback**: Largest plausible amount anywhere (capped at $99,999)

In [ ]:
# Simulate OCR output and test extraction
def make_ocr(lines_with_conf):
    """Create a mock OCRResult from (text, y_position, confidence) tuples."""
    boxes = []
    texts = []
    confs = []
    for text, y, conf in lines_with_conf:
        box = [[10, y], [200, y], [200, y + 20], [10, y + 20]]
        boxes.append(box)
        texts.append(text)
        confs.append(conf)
    return OCRResult(boxes, texts, confs, (500, 300, 3))

# Test Case 1: Clean receipt
ocr1 = make_ocr([
    ("ACME Corp", 10, 0.95),
    ("123 Main Street", 30, 0.90),
    ("Date: 24/01/2024", 60, 0.88),
    ("Item 1     $12.50", 100, 0.85),
    ("Item 2     $8.00", 120, 0.87),
    ("TOTAL      $20.50", 160, 0.92),
])
result1 = FieldExtractor().extract(ocr1)
print("Test 1 - Clean receipt:")
print(f"  Vendor: {result1['vendor']}")
print(f"  Date:   {result1['date']}")
print(f"  Total:  {result1['total']}")
assert result1["vendor"] == "ACME Corp"
assert result1["date"] == "2024-01-24"
assert result1["total"] == "20.50"
print("  ✓ All correct\n")

In [ ]:
# Test Case 2: Total on separate line from keyword
ocr2 = make_ocr([
    ("QuickMart", 10, 0.91),
    ("15 Mar 2024", 30, 0.86),
    ("Coffee    3.50", 60, 0.84),
    ("Sandwich  7.00", 80, 0.82),
    ("TOTAL", 120, 0.93),
    ("10.50", 140, 0.88),
])
result2 = FieldExtractor().extract(ocr2)
print("Test 2 - Total on next line:")
print(f"  Vendor: {result2['vendor']}")
print(f"  Date:   {result2['date']}")
print(f"  Total:  {result2['total']}")
assert result2["total"] == "10.50", f"Expected 10.50, got {result2['total']}"
print("  ✓ Adjacent-line lookup worked\n")

In [ ]:
# Test Case 3: Noisy vendor with short fragments
ocr3 = make_ocr([
    ("Berghotel", 10, 0.80),
    ("Grosse Scheidegg", 30, 0.85),
    ("3818 Grindelwald", 50, 0.75),
    ("TOTAL CHF 54.50", 200, 0.90),
])
result3 = FieldExtractor().extract(ocr3)
print("Test 3 - Multi-line vendor:")
print(f"  Vendor: {result3['vendor']}")
print(f"  Total:  {result3['total']}")
print(f"  ✓ Vendor merge: {'Berghotel' in result3['vendor']}\n")

In [ ]:
# Test Case 4: Phone number should not be extracted as total
ocr4 = make_ocr([
    ("City Diner", 10, 0.92),
    ("Tel: 555-1234", 30, 0.88),
    ("Date 2024-06-15", 50, 0.85),
    ("Burger     12.99", 80, 0.87),
    ("Fries       4.50", 100, 0.86),
    ("Total      17.49", 140, 0.91),
])
result4 = FieldExtractor().extract(ocr4)
print("Test 4 - Phone number filtering:")
print(f"  Vendor: {result4['vendor']}")
print(f"  Total:  {result4['total']}")
assert result4["total"] == "17.49", f"Expected 17.49, got {result4['total']}"
print("  ✓ Phone number correctly ignored\n")

In [ ]:
# Test Case 5: Various date formats
date_tests = [
    ("Date: 24/01/2024", "2024-01-24"),
    ("Date: 2024-03-15", "2024-03-15"),
    ("15 Mar 2024", "2024-03-15"),
    ("March 15, 2024", "2024-03-15"),
    ("Date: 01.06.2024", "2024-06-01"),
]

print("Test 5 - Date format handling:")
ext = FieldExtractor()
for text, expected in date_tests:
    ocr = make_ocr([("Store", 10, 0.9), (text, 30, 0.85), ("Total 5.00", 80, 0.9)])
    result = ext.extract(ocr)
    status = "✓" if result["date"] == expected else "✗"
    print(f"  {status} '{text}' → {result['date']} (expected {expected})")

## 3. Currency Pattern Coverage

The extractor handles multiple currency formats:

| Pattern | Example | Extracted |
|---|---|---|
| Dollar | `$42.50` | `42.50` |
| Euro | `€15.00` | `15.00` |
| Pound | `£8.99` | `8.99` |
| CHF | `CHF 54.50` | `54.50` |
| Decimal only | `123.45` | `123.45` |
| Comma thousands | `1,234.56` | `1234.56` |

Non-monetary patterns (phone numbers, registration numbers) are filtered out
using the `NON_MONETARY_LINE` regex.

In [ ]:
from field_extractor import MONEY_PATTERN, NON_MONETARY_LINE, SKIP_VENDOR_PATTERNS

# Test money pattern coverage
money_tests = [
    "$42.50", "€15.00", "£8.99", "CHF 54.50",
    "123.45", "1,234.56", "RM 99.00", "Rs. 500.00",
]
print("Money pattern matches:")
for text in money_tests:
    match = MONEY_PATTERN.search(text)
    result = (match.group(1) or match.group(2) or match.group(3)) if match else None
    print(f"  '{text}' → {result}")

# Test non-monetary filtering
print("\nNon-monetary line detection:")
non_money = ["Tel: 555-1234", "Fax: 555-5678", "Reg No. 12345", "GST: ABC123"]
for text in non_money:
    filtered = bool(NON_MONETARY_LINE.search(text))
    print(f"  '{text}' → filtered={filtered}")

# Test vendor skip patterns
print("\nVendor skip patterns:")
skip_tests = ["RECEIPT", "Invoice #123", "Berghotel", "Total", "Cash"]
for text in skip_tests:
    skipped = bool(SKIP_VENDOR_PATTERNS.search(text))
    print(f"  '{text}' → skipped={skipped}")

## 4. Known Limitations

| Issue | Impact | Mitigation |
|---|---|---|
| OCR word fragmentation | Vendor names split across boxes | `width_ths=0.9` + vendor merge logic |
| Rotated/skewed receipts | Poor OCR accuracy | CLAHE + denoise preprocessing |
| Non-English receipts | Limited date/total patterns | Extended currency symbols, flexible regex |
| Handwritten receipts | EasyOCR struggles | Falls back to largest amount heuristic |
| Multi-page documents | Only first page processed | Single-image pipeline by design |